# A2A Festival Mission Control — ADK Agents over A2A

This Colab uses **Google ADK agents exposed through the A2A protocol**.

You will run five specialist ADK agents:

- Vibe Scout
- Logistics Pilot
- Budget Oracle
- Sponsor Matchmaker
- Risk & Accessibility Sentinel

Each specialist is converted into an A2A-compatible server using ADK's `to_a2a()` helper.

Mission Control then discovers their Agent Cards and sends them A2A messages.

In [ ]:
!pip -q install "google-adk[a2a]" "a2a-sdk" uvicorn httpx nest_asyncio rich

## 1. Configure Gemini API access

This version uses real ADK `Agent` objects, so you need a Gemini API key.

In Colab, paste your key when prompted. The key is only stored in the current runtime.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Paste your GOOGLE_API_KEY: ")

# ADK can use Gemini through Google AI Studio with this env setup.
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"

print("✅ API key configured for this runtime.")

## 2. Imports and helper runner

In [ ]:
import asyncio
import json
import re
import threading
import time
from typing import Any, Dict, List

import httpx
import nest_asyncio
import uvicorn
from rich.console import Console

from google.adk.agents import Agent
from google.adk.a2a.utils.agent_to_a2a import to_a2a

from a2a.client import A2ACardResolver, ClientConfig, create_client
from a2a.helpers import new_text_message
from a2a.types import MessageSendParams, SendMessageRequest

nest_asyncio.apply()
console = Console()

def run_async(coro):
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(coro)

## 3. Create five real ADK specialist agents

Each one is an ADK `Agent`.  
The deterministic tools make the notebook easier to debug, while ADK decides when and how to use them.

In [ ]:
def extract_budget(text: str) -> int:
    match = re.search(r"£\s?([0-9,]+)|budget[:=]\s*([0-9,]+)", text, flags=re.I)
    if not match:
        return 18000
    raw = next((x for x in match.groups() if x), "18000")
    return int(raw.replace(",", ""))


def extract_theme(text: str) -> str:
    match = re.search(r"theme[:=]\s*([^.;\n]+)", text, flags=re.I)
    return match.group(1).strip() if match else "mythic underwater arcade"


def design_guest_experience(brief: str) -> str:
    """Create immersive festival experience ideas from a brief."""
    theme = extract_theme(brief)
    return f"""
VIBE SCOUT CONCEPT
- Signature entrance: a glowing '{theme}' portal with QR-code story cards.
- Hero moment: a 7-minute projection show every hour.
- Social hook: three hidden photo quests that unlock a secret dessert stall.
- Sound design: future jazz near food, high-energy DJ pocket near the stage.
- Memory anchor: guests leave with a collectible token tied to the story world.
""".strip()


def plan_logistics(brief: str) -> str:
    """Review crowd flow, staffing, schedule, accessibility, and fallback plans."""
    return """
LOGISTICS PILOT CHECK
- Use one-way crowd loops around food and activity zones.
- Keep first-aid visible and away from the loudest activation.
- Stagger headline moments at :15 past each hour to reduce entry surges.
- Add covered queue lanes and no-slip route markers for wet weather.
- Keep accessibility routes flat, visible, and separated from vendor queues.
""".strip()


def plan_budget(brief: str) -> str:
    """Create a practical budget split from the event brief."""
    budget = extract_budget(brief)
    split = {
        "lighting and set pieces": 0.26,
        "performers and hosts": 0.20,
        "vendor support": 0.15,
        "safety and staffing": 0.17,
        "content and social capture": 0.10,
        "contingency": 0.12,
    }
    lines = ["BUDGET ORACLE PLAN"]
    for item, pct in split.items():
        lines.append(f"- {item}: £{budget * pct:,.0f}")
    lines.append("- Rule: protect safety, staffing, and contingency before increasing spectacle spend.")
    return "\\n".join(lines)


def match_sponsors(brief: str) -> str:
    """Suggest sponsor categories and activation ideas for the event."""
    return """
SPONSOR MATCHMAKER PLAN
- Indie game studio sponsor: playable demo zone with branded quest cards.
- Sustainable drinks partner: reusable cup system with collectible cup art.
- Local art school partner: student projection-mapping showcase.
- Audio brand partner: headphone listening cave for quieter immersion.
- Sponsor rule: every sponsor must improve the guest experience, not just place logos.
""".strip()


def audit_risk_accessibility(brief: str) -> str:
    """Identify risk, safety, accessibility, and inclusion issues."""
    return """
RISK & ACCESSIBILITY SENTINEL
- Risk: crowd bottlenecks near food. Fix with pre-order QR lanes and separate pickup flow.
- Risk: sensory overload. Fix with a quiet decompression zone and clear signage.
- Risk: weather. Fix with covered queues and weighted outdoor structures.
- Accessibility: step-free route, accessible toilets, seating pockets, large-print map.
- Inclusion: keep quests playable without smartphones by offering paper cards.
""".strip()

In [ ]:
MODEL = "gemini-2.0-flash"

vibe_scout_agent = Agent(
    model=MODEL,
    name="vibe_scout",
    description="Creates memorable guest experiences for pop-up festivals.",
    instruction="""
You are Vibe Scout, an event-experience designer.
Use the design_guest_experience tool.
Return concise, vivid, practical ideas.
""",
    tools=[design_guest_experience],
)

logistics_pilot_agent = Agent(
    model=MODEL,
    name="logistics_pilot",
    description="Plans event logistics, crowd flow, staffing, and fallback operations.",
    instruction="""
You are Logistics Pilot, an event operations lead.
Use the plan_logistics tool.
Return practical operational checks and fixes.
""",
    tools=[plan_logistics],
)

budget_oracle_agent = Agent(
    model=MODEL,
    name="budget_oracle",
    description="Creates budget splits and trade-off rules for events.",
    instruction="""
You are Budget Oracle, a pragmatic budget planner.
Use the plan_budget tool.
Return a clear budget split and trade-off rule.
""",
    tools=[plan_budget],
)

sponsor_matchmaker_agent = Agent(
    model=MODEL,
    name="sponsor_matchmaker",
    description="Finds sponsor categories that improve the event experience.",
    instruction="""
You are Sponsor Matchmaker.
Use the match_sponsors tool.
Suggest sponsors that make the event better, not just more commercial.
""",
    tools=[match_sponsors],
)

risk_accessibility_agent = Agent(
    model=MODEL,
    name="risk_accessibility_sentinel",
    description="Audits event safety, inclusion, accessibility, and operational risk.",
    instruction="""
You are Risk & Accessibility Sentinel.
Use the audit_risk_accessibility tool.
Identify risks and give direct fixes.
""",
    tools=[audit_risk_accessibility],
)

ADK_AGENTS = [
    ("Vibe Scout", vibe_scout_agent, 10101),
    ("Logistics Pilot", logistics_pilot_agent, 10102),
    ("Budget Oracle", budget_oracle_agent, 10103),
    ("Sponsor Matchmaker", sponsor_matchmaker_agent, 10104),
    ("Risk & Accessibility Sentinel", risk_accessibility_agent, 10105),
]

console.print("✅ Created five ADK agents.")
for label, agent, port in ADK_AGENTS:
    console.print(f"- {label}: ADK name={agent.name}, port={port}")

## 4. Expose each ADK agent through A2A

This is the key ADK + A2A step.

`to_a2a(agent, port=...)` turns an ADK agent into an A2A-compatible app with an Agent Card and A2A routes.

In [ ]:
def run_a2a_server(label: str, adk_agent: Agent, port: int):
    app = to_a2a(adk_agent, port=port)
    uvicorn.run(app, host="127.0.0.1", port=port, log_level="error")


threads = []
for label, adk_agent, port in ADK_AGENTS:
    thread = threading.Thread(
        target=run_a2a_server,
        args=(label, adk_agent, port),
        daemon=True,
    )
    thread.start()
    threads.append(thread)

time.sleep(4)

console.print("✅ ADK agents are now exposed as A2A servers:")
for label, _, port in ADK_AGENTS:
    console.print(f"- {label}: http://127.0.0.1:{port}")

## 5. Discover Agent Cards

Mission Control fetches each A2A Agent Card from the well-known endpoint.

In [ ]:
async def discover_cards():
    discovered = []

    async with httpx.AsyncClient(timeout=30) as httpx_client:
        for label, _, port in ADK_AGENTS:
            base_url = f"http://127.0.0.1:{port}"
            resolver = A2ACardResolver(
                httpx_client=httpx_client,
                base_url=base_url,
            )
            card = await resolver.get_agent_card()
            discovered.append((label, card))

    return discovered


discovered_cards = run_async(discover_cards())

for label, card in discovered_cards:
    console.rule(label)
    console.print(card)

## 6. Send A2A messages to each ADK agent

Mission Control uses an A2A client to call every remote ADK agent.

In [ ]:
def compact_event_text(event: Any) -> str:
    """Extract readable text from A2A client events without relying on one SDK shape."""
    raw = str(event)

    # Try common artifact text patterns first.
    for key in ["text='", 'text="']:
        if key in raw:
            after = raw.split(key, 1)[1]
            quote = "'" if key.endswith("'") else '"'
            return after.split(quote, 1)[0].replace("\\n", "\n")

    return raw


async def send_to_agent(card, text: str) -> str:
    client_config = ClientConfig(streaming=False)
    client = await create_client(agent=card, client_config=client_config)

    message = new_text_message(text)
    request = SendMessageRequest(params=MessageSendParams(message=message))

    chunks = []
    async for event in client.send_message(request):
        chunks.append(compact_event_text(event))

    close = getattr(client, "close", None)
    if close:
        maybe = close()
        if asyncio.iscoroutine(maybe):
            await maybe

    cleaned = [c for c in chunks if c and "TASK_STATE_WORKING" not in c]
    return "\n".join(cleaned[-3:]) if cleaned else "\n".join(chunks)


async def mission_control(brief: str) -> str:
    card_by_label = {label: card for label, card in discovered_cards}

    tasks = {
        "Vibe Scout": f"Design the guest experience for this event:\n{brief}",
        "Logistics Pilot": f"Plan the logistics and crowd flow for this event:\n{brief}",
        "Budget Oracle": f"Create the budget split for this event:\n{brief}",
        "Sponsor Matchmaker": f"Suggest sponsors and activation partners for this event:\n{brief}",
        "Risk & Accessibility Sentinel": f"Audit safety, accessibility, and inclusion for this event:\n{brief}",
    }

    results = {}
    for label, task in tasks.items():
        console.print(f"➡️ Sending A2A message to {label}")
        results[label] = await send_to_agent(card_by_label[label], task)

    sections = [
        "# Festival Mission Control Plan",
        "",
        "## Original brief",
        brief.strip(),
        "",
        "## Specialist A2A reports",
    ]

    for label, result in results.items():
        sections.append(f"\n### {label}\n{result}")

    sections.append("""
## Mission Control synthesis

- Build the guest journey around the glowing portal entrance, hourly projection moment, and hidden quest mechanic.
- Use the logistics loop as the spine of the experience so crowd movement feels intentional.
- Protect safety, accessibility, staffing, and contingency before expanding spectacle.
- Let sponsors power useful moments: demos, reusable cups, audio zones, and student art showcases.
- Include both digital and paper quest paths so guests without smartphones can still participate.
""")

    return "\n".join(sections)

## 7. Run the full ADK + A2A mission

This calls all five ADK agents over A2A and combines their reports.

In [ ]:
brief = """
Plan a one-night pop-up festival for 500 people in East London.
Theme: mythic underwater arcade.
Budget: £18000.
Audience: students, indie game developers, and local artists.
Constraints: low waste, high social shareability, safe crowd flow, and accessible participation.
"""

plan = run_async(mission_control(brief))
print(plan)

## 8. What makes this properly ADK + A2A?

This notebook uses:

- Google ADK `Agent`
- ADK tools
- ADK `to_a2a()` conversion
- A2A Agent Card discovery
- A2A client message sending
- Five remote ADK agents exposed over A2A
- One Mission Control orchestrator consuming those agents

This is no longer a mock FastAPI example.